# Milky Way Environment Walkthrough

This notebook recalculates the Milky Way density and velocity values listed in Table 1 of Saricaoglu and Daylan, *Captured Primordial Black Holes in Microlensing Surveys*. The calculations use Captus's built-in Milky Way model and Astropy's unit conversions.

In [ ]:
import sys

import numpy as np
import pandas as pd
from astropy import constants as const
from astropy import units as u
from tdpy.verbosity import print

import captus
from captus.utils.misc import get_MW_params

print(f"Python {sys.version.split()[0]} | Captus {captus.__version__}")
radii_kpc = np.array([8.0, 2.2, 0.05])  # [kpc]

## Recalculate Table 1

For each Galactocentric radius, Captus returns the dark-matter density in solar masses per cubic kiloparsec, circular speed, and one-dimensional velocity dispersion. Astropy handles the density-to-energy-density conversion.

In [ ]:
def build_environment_table(radii):
    rows = []
    for radius_kpc in radii:
        params = get_MW_params(float(radius_kpc))
        density_gev_cm3 = (
            params["DM_density"].to(u.kg / u.m**3) * const.c**2
        ).to_value(u.GeV / u.cm**3)
        rows.append(
            {
                "r [kpc]": radius_kpc,
                "rho_DM [GeV cm^-3]": density_gev_cm3,
                "v_c [km s^-1]": params["v_circ"],
                "sigma_DM [km s^-1]": params["DM_dispersion"],
            }
        )
    return pd.DataFrame(rows)


environment_table = build_environment_table(radii_kpc)
display(environment_table.round(3))


In [ ]:
expected_vcirc = np.array([233.20, 200.87, 77.40])  # [km/s]
expected_sigma = np.array([164.89, 142.04, 54.73])  # [km/s]
np.testing.assert_allclose(environment_table["v_c [km s^-1]"], expected_vcirc, atol=0.01)
np.testing.assert_allclose(environment_table["sigma_DM [km s^-1]"], expected_sigma, atol=0.01)
np.testing.assert_allclose(environment_table["rho_DM [GeV cm^-3]"].iloc[-1], 3.195, atol=0.001)

density_values = environment_table["rho_DM [GeV cm^-3]"].to_numpy()
print(f"Density array shape={density_values.shape}, dtype={density_values.dtype}")
print(f"Inner-Galaxy density: {density_values[-1]:.3f} GeV cm^-3")


## Reproducible velocity illustration

This is a synthetic illustration of the paper's truncated Maxwellian speed distribution at 8 kpc, not a sample from the paper's encounter simulations. Three Gaussian velocity components produce a Maxwell speed distribution; samples above the adopted 550 km s$^{-1}$ escape speed are discarded.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from scipy.stats import maxwell

sigma_dm_kms = float(environment_table.loc[0, "sigma_DM [km s^-1]"])
escape_speed_kms = 550.0  # [km/s]
seed = 20261014
rng = np.random.default_rng(seed)
velocity_components = rng.normal(0.0, sigma_dm_kms, size=(100_000, 3))  # [km/s]
speeds_kms = np.linalg.norm(velocity_components, axis=1)
speeds_kms = speeds_kms[speeds_kms <= escape_speed_kms]

repeat_rng = np.random.default_rng(seed)
repeat_components = repeat_rng.normal(0.0, sigma_dm_kms, size=(100_000, 3))
np.testing.assert_array_equal(velocity_components, repeat_components)
print(f"Retained {speeds_kms.size:,} of 100,000 draws below escape speed.")

speed_grid_kms = np.linspace(0, escape_speed_kms, 400)
speed_pdf = maxwell(scale=sigma_dm_kms)
truncated_pdf = speed_pdf.pdf(speed_grid_kms) / speed_pdf.cdf(escape_speed_kms)
figure, axis = plt.subplots(figsize=(7, 4), layout="constrained")
axis.hist(speeds_kms, bins=50, density=True, color="tab:blue", alpha=0.55, label="Seeded toy sample")
axis.plot(speed_grid_kms, truncated_pdf, color="black", label="Truncated Maxwellian")
axis.set_xlabel(r"Speed [km s$^{-1}$]")
axis.set_ylabel(r"Probability density [(km s$^{-1}$)$^{-1}$]")
axis.legend(frameon=False)
axis.grid(False)

output_dir = Path("paper_examples")
output_dir.mkdir(parents=True, exist_ok=True)
figure_path = output_dir / "milky_way_velocity_distribution.png"
print(f"Writing to {figure_path}...")
figure.savefig(figure_path, dpi=180)
display(figure)
plt.close(figure)


## Export

The velocity figure is saved beneath `paper_examples/` relative to the notebook's working directory. Export this notebook from Jupyter, or run `jupyter nbconvert --to html examples/paper_environment.ipynb` from the repository root.